# Reproducing Yemini et al. (2021), NeuroPAL, from dandiset 000715

**Paper.** Yemini E, Lin A, Nejatbakhsh A, Varol E, Sun R, Mena GE, Samuel ADT, Paninski L,
Venkatachalam V, Hobert O (2021). NeuroPAL: A Multicolor Atlas for Whole-Brain Neuronal
Identification in *C. elegans*. *Cell* 184:272-288.
[doi:10.1016/j.cell.2020.12.012](https://doi.org/10.1016/j.cell.2020.12.012). The claims file
was written from the accepted author manuscript in PubMed Central (PMC10494711) with Methods S1
and S2; the version of record was not read.

**Dandiset.** [DANDI:000715](https://dandiarchive.org/dandiset/000715/0.241009.1514), version
0.241009.1514 (doi:10.48324/dandi.000715/0.241009.1514). The paper cites no DANDI version (its
data statement points to Zenodo). There are four published versions (0.240424.1847,
0.240611.1953, 0.240614.1942, 0.241009.1514) and the draft; all list the same ten assets with the
same asset and blob identifiers and sizes, so their content is identical. The most recent
published version is pinned.

**What the dandiset holds.** Ten NWB files, each one worm head: a five-channel volume and the
hand-annotated neurons (one centre voxel and one name per neuron). The data check below shows
that these are the heads of the paper's ten atlas worms (otIs669), whose head and tail volumes
are also on Zenodo (doi:10.5281/zenodo.3906530); the tails are not in the dandiset. The files'
own metadata say otherwise (strain OH16230, a spinning disk confocal, a microfluidic chip).

**Outside inputs.** The NeuroPAL colors of each neuron, from `data.mat` of the paper's
Optimal_Coloring code (github.com/Eviatar/Optimal_Coloring at 990c597), for
ms2-neuropal-color-violations (and the reporter table of the same file for fig7de, in
`optimal_coloring.ipynb`). Two head volumes of the Zenodo record, read only to check what the
dandiset's files are, and the hand annotations of all ten worms (`*_ID.mat`), compared with the
dandiset's. The ganglion of each neuron, from the CELL_ID neuron tables (Data/NeuroPAL/herm_data.mat
at d9f5df9), for the head part of fig7bc-ventral-ganglion-lowest.

| Claim | Status here |
|---|---|
| fig2cd-variability-differs | tested, head only |
| fig2c-ollr-more-variable-than-olll | tested |
| fig7bc-identification-accuracy | tested, head only |
| fig7bc-ventral-ganglion-lowest | blocked: tails not in the dandiset; head part reported |
| fig7b-eight-labels-above-90 | tested |
| ms1-uncertain-first-faster | tested, head only |
| ms1-position-only-accuracy | tested, head only |
| ms1-detection-accuracy | tested, head only |
| ms1-calibration | tested, head only |
| ms2-neuropal-color-violations | tested, head atlas (with outside colors) |
| fig7de-optimal-coloring-violations | tested in `optimal_coloring.ipynb`, head atlas (with outside reporters) |
| other claims (figures 1, 3 to 6, S1, S2) | not tested on the dandiset; see the record |

In [ ]:
import time
import io
import warnings

import numpy as np
import pandas as pd
import requests
import scipy.io as sio
from scipy import stats
import matplotlib.pyplot as plt

import neuropal as npl

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 300)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
worms = npl.load_all()
print(f"loaded {len(worms)} files from dandiset {npl.DANDISET_ID} version {npl.DANDISET_VERSION} "
      f"in {time.time() - started:.0f} s")

## Data check

In [ ]:
info = pd.DataFrame([{
    "subject": w["subject"], "location": w["location"], "strain": w["strain"],
    "device": w["device"][0], "chip": w["subject_description"], "born": w["date_of_birth"][:10],
    "session": w["session_start"][:10], "shape": w["shape"], "grid_um": tuple(np.round(w["grid"], 4)),
    "rois": len(w["labels"]), "voxels_per_roi": w["voxels_per_roi"].tolist(),
    "named": sum(1 for l in w["labels"] if l), "unnamed": sum(1 for l in w["labels"] if not l),
    "duplicate_names": len([l for l in set(w["labels"]) if l and w["labels"].count(l) > 1]),
} for w in worms])
print(info.to_string(index=False))
print("\nChannel order in every file:", worms[0]["channels"], "; RGBW indices:", worms[0]["rgbw"].tolist())
print("Paper (Methods S1, MICCAI atlas paper): 10 young-adult otIs669 (OH15262) worms, head and tail, "
      "Zeiss LSM 880, about 0.2 x 0.2 x 0.8 um voxels, 195 head neurons annotated per worm.")

Every file is a head (`location`), with one voxel per annotated neuron. The files say strain
OH16230 (the NeuroPAL;GCaMP6s strain of the whole-brain recordings), a spinning disk confocal, a
worm "in microfluidic chip", birth on 2021-01-05 and imaging on 2021-01-07, the same for all ten.
The paper's atlas worms are otIs669 (OH15262), imaged on a Zeiss LSM 880, and the paper was
published in January 2021. The subject identifiers (7_YAaLR, 38_YAaDV, ...) are those of the ten
otIs669 worms in the Zenodo record. The next cell compares two volumes with the Zenodo copies.

In [ ]:
import fsspec, zipfile, h5py, remfile

ZENODO_ZIP = "https://zenodo.org/records/3906530/files/NeuroPAL%20data.zip?download=1"
zf = zipfile.ZipFile(fsspec.open(ZENODO_ZIP, block_size=2**22).open())
members = {m.split("/")[-1][:-4]: m for m in zf.namelist() if m.endswith(".mat") and "YAa" in m and "_ID" not in m}
print("Zenodo head volumes (otIs669 Young Adults):", sorted(members))
print("Dandiset subjects:", sorted(w["subject"] for w in worms))
print("Same set of identifiers:", sorted(members) == sorted(w["subject"] for w in worms))
for subject in ["7_YAaLR", "76_YAaDV"]:
    m = sio.loadmat(io.BytesIO(zf.read(members[subject])), squeeze_me=True, struct_as_record=False,
                    variable_names=["data", "info"])
    zen = m["data"]
    w = next(w for w in worms if w["subject"] == subject)
    asset = next(a for a in npl.list_assets() if subject in a["path"].replace("-", "_"))
    with h5py.File(remfile.File(asset["url"]), "r") as f:
        dan = f["acquisition/NeuroPALImageRaw/data"][()]
    same = zen.shape == tuple(np.array(dan.shape)[[1, 0, 2, 3]]) and np.array_equal(np.transpose(zen, (1, 0, 2, 3)), dan)
    print(f"{subject}: Zenodo {zen.shape}, dandiset {dan.shape}; identical voxel values after swapping x and y: {same}; "
          f"Zenodo scale {np.round(m['info'].scale, 4)} against dandiset grid {np.round(w['grid'], 4)}; "
          f"Zenodo RGBW (1-based) {m['info'].RGBW}, DIC/GFP channel (1-based) {m['info'].DIC}")
    del zen, dan

# The annotations of all ten worms against the Zenodo *_ID.mat files (names and positions); reading
# the other eight volumes in full would take about 100 s each, so they are compared by annotations.
ids = {m.split("/")[-1][:-7]: m for m in zf.namelist() if m.endswith("_ID.mat") and "YAa" in m}
rows = []
for w in worms:
    zlab, zpos = npl.zenodo_annotations(zf, ids[w["subject"]])
    zvox = zpos[:, [1, 0, 2]] - 1                      # MATLAB (y, x, z), 1-based, to the dandiset's (x, y, z), 0-based
    same_names = sorted(zlab) == sorted(w["labels"])
    order = [zlab.index(l) for l in w["labels"] if zlab.count(l) == 1]
    stored = np.array([w["vox"][i] for i, l in enumerate(w["labels"]) if zlab.count(l) == 1])
    diff = np.abs(zvox[order] - stored)
    rows.append(dict(subject=w["subject"], view=w["subject"][-2:], zenodo_neurons=len(zlab), dandiset_neurons=len(w["labels"]),
                     same_names=same_names, compared_by_unique_name=len(order), max_abs_diff_voxels=round(float(diff.max()), 3),
                     equal_to_rounded_zenodo=bool((np.round(zvox[order]) == stored).all())))
print("\nAnnotations against the Zenodo *_ID.mat files:")
print(pd.DataFrame(rows).to_string(index=False))


The two volumes checked are identical, voxel for voxel, to the Zenodo volumes of the otIs669 atlas
worms. For all ten worms the annotations hold the same names as the Zenodo `*_ID.mat` files, and
every stored voxel equals the Zenodo position, made 0-based and rounded (largest difference 0.5
voxel), comparing the neurons whose name occurs once. So the dandiset holds the heads of the paper's ten
atlas worms, and its strain, device, subject description and dates are wrong. The Zenodo files
mark the third channel (1-based) as the DIC (transmitted light) channel, which the dandiset calls
"GFP-GCaMP"; the next cell shows its values.

In [ ]:
ch = pd.DataFrame([dict(subject=w["subject"], **{f"mean_{c}": round(float(v)) for c, v in zip(w["channels"], w["channel_mean"])},
                        **{f"max_{c}": int(v) for c, v in zip(w["channels"], w["channel_range"][1])}) for w in worms])
print(ch.to_string(index=False))
print("\nPanneuronal TagRFP-T (z-scored, 3x3x1 patch) at the stored voxel and shifted by -1 or +1 voxel on every axis:")
print(pd.DataFrame([dict(subject=w["subject"], **{f"offset_{k}": round(v, 2) for k, v in w["white_at_offset"].items()}) for w in worms]).to_string(index=False))
counts = pd.Series([l for w in worms for l in set(w["labels"]) if l]).value_counts()
print(f"\n{counts.size} distinct neuron names; held by all 10 worms: {(counts == 10).sum()}; by 8 or 9: {counts.between(8, 9).sum()}; "
      f"by fewer than 8: {(counts < 8).sum()} ({', '.join(sorted(counts[counts < 8].index))})")
pos = pd.DataFrame([dict(subject=w["subject"], **{f"{a}_range_um": f"{w['pos'][:, i].min():.1f}-{w['pos'][:, i].max():.1f}" for i, a in enumerate('xyz')},
                         **{f"{c}_range": f"{w['rgbw_colors'][:, i].min():.2f}-{w['rgbw_colors'][:, i].max():.2f}" for i, c in enumerate('RGBW')}) for w in worms])
print("\nNeuron positions (um = voxel x grid) and colors (median z-score in the patch):")
print(pos.to_string(index=False))
sig = [(w["shape"], round(float(w["channel_mean"].sum()), 1), len(w["labels"])) for w in worms]
print("\nPairs of files with the same volume shape, channel means and neuron count (possible duplicates):",
      [(worms[i]["subject"], worms[j]["subject"]) for i in range(10) for j in range(i + 1, 10) if sig[i] == sig[j]])

The third channel has a mean near 20,000 in every file, far above the fluorescence channels, as
expected of transmitted light, not of GCaMP. The panneuronal channel is brighter at the stored
voxel than one voxel lower on every axis and about as bright as one voxel higher, so the voxel
indices are read as 0-based centres, which the comparison with the Zenodo annotations above confirms. Positions span about 150 to 200 um along x (the head's
length), as expected. No two files share shape, intensities and neuron count, so no recording is
deposited twice. These are static volumes with no time axis or trial table, so there is no clock
to check. About 3% of the stored neurons per worm have no name (empty label); they are left out.
The names cover 198 neurons, of which 168 are named in all ten worms; the paper reports 195 head
neurons annotated in each worm.

## The statistical atlas (MS1 Algorithm 4)

The atlas is fit on the named neurons: position (um) and the R, G, B landmark colors (mNeptune2.5,
CyOFP1, mTagBFP2) read from the z-scored volume. Algorithm 4 as written alternates an affine
regression of the atlas means on each worm (equations 4.15 and 4.16) with the moments of the
aligned worms (equations 4.13 and 4.14) until the change is below a tolerance. On these data it
does not converge: the atlas is defined only up to an affine map, and the regression shrinks it
along its minor axes at every alternation (table below). The paper does not say how this was
avoided. The atlas here keeps the gauge of the initial worm (after each update the atlas is mapped
back, by the affine map that best fits its means to the initial worm's neurons) and stops after
`npl.ATLAS_ITER` = 20 alternations, where the position block has settled. This is an assumption;
the results are shown under other choices.

In [ ]:
feats = [npl.worm_features(w) for w in worms]

def atlas_summary(names, mu, sigma):
    coeff, m = npl._pca(mu[:, :3])
    P = (mu[:, :3] - m) @ coeff
    sd = np.sqrt(np.einsum("ai,nab,bi->ni", coeff, sigma[:, :3, :3], coeff))
    csd = np.sqrt(np.diagonal(sigma[:, 3:, 3:], axis1=1, axis2=2))
    return P.max(0) - P.min(0), np.median(sd, 0), np.median(csd, 0)

rows = []
for gauge in (False, True):
    for it in (1, 5, 20, 100, 300):
        names, mu, sigma, inf = npl.fit_atlas(feats, seed=0, max_iter=it, gauge=gauge)
        ext, sd, csd = atlas_summary(names, mu, sigma)
        rows.append(dict(gauge=gauge, iterations=it, converged=inf["converged"], change_mu=round(inf["dmu"], 4),
                         change_sigma=round(inf["dsig"], 4), extent_axes_um=np.round(ext, 1).tolist(),
                         median_sd_axes_um=np.round(sd, 2).tolist(), median_color_sd=np.round(csd, 2).tolist()))
print(pd.DataFrame(rows).to_string(index=False))
names, mu, sigma, atlas_info = npl.fit_atlas(feats, seed=0)
print(f"\nAtlas used: {len(names)} neurons, initial worm {worms[atlas_info['init_worm']]['subject']} (seed 0), "
      f"{atlas_info['iterations']} alternations, gauge fixed; worms per neuron: min {atlas_info['counts'].min()}, "
      f"median {int(np.median(atlas_info['counts']))}")

Without the gauge the extent of the atlas along its second and third axes falls from about 22 and
16 um to 0 by 300 alternations; with it the position block holds (extents within a few percent
from 5 to 300 alternations) while the color covariances keep growing slowly. The tolerance of 1e-6
is never reached.

### fig2cd-variability-differs

> In a probabilistic atlas built from 10 young-adult NeuroPAL worms, the positional variability of
> cell bodies differs across neuron types.

Criterion: per neuron, the standard deviation of its positional covariance along each principal
axis of the atlas means; reproduced when along at least one axis the largest is more than twice
the smallest, within the head and within the tail. Only heads are in the dandiset.

In [ ]:
def axis_sd(mu, sigma):
    coeff, m = npl._pca(mu[:, :3])
    return np.sqrt(np.einsum("ai,nab,bi->ni", coeff, sigma[:, :3, :3], coeff)), coeff

def sd_table(mu, sigma, counts=None):
    sd, _ = axis_sd(mu, sigma)
    keep = np.ones(len(sd), bool) if counts is None else counts >= 8
    s = sd[keep]
    return s, pd.DataFrame({"axis": ["1 (major, anterior-posterior)", "2", "3"], "min_sd_um": s.min(0).round(3),
                            "median_sd_um": np.median(s, 0).round(3), "max_sd_um": s.max(0).round(3),
                            "max_over_min": (s.max(0) / s.min(0)).round(2),
                            "p90_over_p10": (np.percentile(s, 90, 0) / np.percentile(s, 10, 0)).round(2)})

s_all, t_all = sd_table(mu, sigma)
print(f"All {len(names)} atlas neurons (the ridge of {npl.RIDGE} um^2 bounds each SD below by {np.sqrt(npl.RIDGE):.2f} um):")
print(t_all.to_string(index=False))
s8, t8 = sd_table(mu, sigma, atlas_info["counts"])
print(f"\nNeurons named in at least 8 worms ({len(s8)}):")
print(t8.to_string(index=False))
print("\nVariants (neurons in at least 8 worms), largest over smallest SD per axis:")
for label, kw in [("initial worm 11_YAaLR", dict(init=0)), ("initial worm 38_YAaDV", dict(init=2)), ("5 alternations", dict(max_iter=5)),
                  ("100 alternations", dict(max_iter=100)), ("no gauge, 20 alternations", dict(gauge=False)),
                  ("forward regression (eq 4.11)", dict(direction="forward"))]:
    n_, m_, s_, i_ = npl.fit_atlas(feats, **kw)
    sv, tv = sd_table(m_, s_, i_["counts"])
    print(f"  {label:28s} initial worm {worms[i_['init_worm']]['subject']:9s} ratios {tv.max_over_min.tolist()}  p90/p10 {tv.p90_over_p10.tolist()}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for i, ax in enumerate(axes):
    ax.hist(s8[:, i], bins=30, color="0.4")
    ax.set_xlabel(f"SD along axis {i + 1} (um)")
axes[0].set_ylabel("neurons")
plt.tight_layout()
plt.show()
print("Plotted ranges (um):", [(round(s8[:, i].min(), 2), round(s8[:, i].max(), 2)) for i in range(3)], "; no point outside the axes")

**Result: partially reproduced: the head half holds, and the tail half cannot be tested, as the
tails are not in the dandiset.** Among the 191 neurons named in at least eight worms, the largest
per-neuron standard deviation is 36.1, 21.2 and 13.1 times the smallest along the three axes (90th
over 10th percentile 8.3, 3.2 and 3.7). Every variant of the atlas fit, including the forward
regression of equation 4.11, gives ratios of 9.39 or more along every axis.

### fig2c-ollr-more-variable-than-olll

> In the head atlas, OLLR shows over twice the positional variability of OLLL along the
> anterior-posterior axis.

In [ ]:
def ollr_olll(names, mu, sigma):
    sd, coeff = axis_sd(mu, sigma)
    out = {}
    iR, iL = names.index("OLLR"), names.index("OLLL")
    out["a_sd_ratio"] = sd[iR, 0] / sd[iL, 0]
    out["b_var_ratio"] = out["a_sd_ratio"] ** 2
    r50 = np.sqrt(stats.chi2.ppf(0.5, 2))
    for plane, keep, q in (("axes_1_2", [0, 1], 2), ("axes_1_3", [0, 2], 1)):
        ext = []
        for i in (iR, iL):
            S = coeff.T @ sigma[i, :3, :3] @ coeff            # covariance in the principal frame
            Spp = S[np.ix_(keep, keep)]
            Spq = S[np.ix_(keep, [q])]
            cond = Spp - Spq @ Spq.T / S[q, q]                 # conditional 2-D covariance in the plane
            ext.append(2 * r50 * np.sqrt(cond[0, 0]))          # AP extent of the 50% contour
        out[f"c_extent_ratio_{plane}"] = ext[0] / ext[1]
    out["sd_OLLR_um"], out["sd_OLLL_um"] = sd[iR, 0], sd[iL, 0]
    return out

rows = [dict(atlas="main (initial worm 70_YAaLR, gauge, 20)", **ollr_olll(names, mu, sigma))]
for label, kw in [("initial worm 11_YAaLR", dict(init=0)), ("initial worm 38_YAaDV", dict(init=2)), ("5 alternations", dict(max_iter=5)),
                  ("100 alternations", dict(max_iter=100)), ("no gauge, 20 alternations", dict(gauge=False)),
                  ("forward regression (eq 4.11)", dict(direction="forward"))]:
    n_, m_, s_, i_ = npl.fit_atlas(feats, **kw)
    rows.append(dict(atlas=label, **ollr_olll(n_, m_, s_)))
print(pd.DataFrame(rows).round(3).to_string(index=False))
print("\nWorms naming OLLR:", atlas_info["counts"][names.index("OLLR")], "; OLLL:", atlas_info["counts"][names.index("OLLL")])

**Result: reproduced.** Along the major axis OLLR's standard deviation is 2.69 um and OLLL's
1.25 um, a ratio of 2.14 (reading a); the variance ratio is 4.60 (reading b) and the extent of the
50% contour gives 1.63 and 2.06 in the two planes (reading c). Under the six variants of the
atlas fit (including the forward regression of equation 4.11) the standard deviation ratio is 2.00
to 2.33 and the variance ratio 4.01 to 5.44.

### ms1-detection-accuracy

> The automatic neuron detection (matching pursuit) reaches 80% accuracy in the head and 88% in
> the tail, at 191 and 44 detections, with precision and recall both about 90% in the head and 93%
> in the tail.

Expected (head): accuracy 80 +/- 8%, precision and recall 90 +/- 9%, means over the worms. The
detection is a port of CELL_ID (filter of gut cells and lysosomes, matching pursuit with a local
fit of a truncated Gaussian, 1.5 um exclusion radius) run for K = 191 on each head; detections
are matched to the hand annotations by the Hungarian algorithm with distances over 3 um excluded.
The hand annotations are read as all stored neurons (main) and as the named ones only.

In [ ]:
t0 = time.time()
detections = npl.detect_all(191)
print(f"detection on 10 heads in {time.time() - t0:.0f} s")
rows = []
for d, w in zip(detections, worms):
    assert d["subject"] == w["subject"]
    keep = [i for i, l in enumerate(w["labels"]) if l]
    for which, ann in (("all stored", w["pos"]), ("named", w["pos"][keep])):
        m = npl.match_detections(d["centres"] * w["grid"], ann)
        rows.append(dict(subject=w["subject"], annotations=which, detections=len(d["centres"]), fits=d["n_fits"], converged=d["n_converged"],
                         n_annotated=len(ann), tp=m["tp"], fp=m["fp"], fn=m["fn"], accuracy=100 * m["accuracy"],
                         precision=100 * m["precision"], recall=100 * m["recall"]))
det_tab = pd.DataFrame(rows)
print(det_tab.round(1).to_string(index=False))
summ = det_tab.groupby("annotations")[["accuracy", "precision", "recall"]].agg(["mean", "std"]).round(1)
print("\nMean and SD over the 10 heads (%):")
print(summ.to_string())
print("\nTolerances: accuracy 72 to 88; precision and recall 81 to 99")
print(f"Gaussian fits whose optimiser (SLSQP) reported success: {sum(d['n_converged'] for d in detections)} of "
      f"{sum(d['n_fits'] for d in detections)}")

**Result: partially reproduced for the head; the tails are not in the dandiset.** Over the ten
heads at K = 191 the mean precision is 83.4% and the mean recall 82.5%, inside 81 to 99%, while
the mean accuracy is 71.0% (SD 4.6), just below the 72 to 88% range. Against the named neurons
only, accuracy is 70.6%, precision 82.5% and recall 82.8%. The optimiser (SLSQP, in place of
MATLAB's fmincon) reported success in 2,043 of 2,064 Gaussian fits; whether fmincon would place
the detections differently is not measured here.

## Leave-one-out identification (MS1 Algorithm 5)

For each held-out worm the atlas is fit on the other nine and the worm's named neurons are
identified. Two alignments are run, as the claims file asks: Algorithm 5 as written (identity
start on the positions as given) and the CELL_ID global alignment (principal axes and a search of
rotations). A third run aligns the held-out worm with its true correspondences; it is not the
paper's method and only bounds what the atlas and the colors allow.

The files carry no orientation of the worm. The subject names end in LR (imaged in left-right
view) or DV (dorsal-ventral view), and the alignment by true correspondences (next cells) shows
that the five DV heads are stored rotated by about 90 degrees about the long axis relative to the
five LR heads. Algorithm 5's identity start cannot undo this, and the CELL_ID rotation search does
not search rotations: as in CELL_ID, the principal-axis fit after each rotation undoes it. A
fourth run, not the paper's method, starts Algorithm 5 from the identity and from rotations of 90,
180 and 270 degrees about the long axis and keeps the run with the lowest assignment cost, using
no annotations ("rotated"). Reading a identifies the
named hand-annotated neurons; reading b identifies the automatic detections of
ms1-detection-accuracy, an annotation counting as right when the detection matched to it within
3 um gets its name. Neurons whose name is not in the nine-worm atlas count as wrong.

In [ ]:
t0 = time.time()
runs = {}
for alignment in ("methods", "code", "rotated", "oracle"):
    for color in (True, False):
        runs[(alignment, color)] = npl.run_loo(feats, alignment, color=color)
print(f"leave-one-out runs done in {time.time() - t0:.0f} s")

def score_rows(res):
    out = {}
    for by_class in (False, True):
        per, pool = npl.scores(res, by_class)
        tag = "class" if by_class else "neuron"
        out[f"mean_per_worm_{tag}"] = 100 * per.mean()
        out[f"pooled_{tag}"] = 100 * pool
    return out

unmatched = [int((r["true"] < 0).sum()) for r in runs[("methods", True)]]
print("Named neurons per held-out worm whose name is not in the nine-worm atlas:", unmatched)
print("Largest difference between this Sinkhorn and CELL_ID's log-space version, per held-out worm:",
      [float(np.abs(r["P"] - npl.sinkhorn_logspace(r["logL"])).max().round(4)) for r in runs[("methods", True)]])

In [ ]:
# Orientation. The subject names end in LR (imaged in left-right view) or DV (dorsal-ventral view).
# For each held-out worm: the extents of its neurons along their principal axes and along the image
# axes, and the rotation part of the alignment to the nine-worm atlas by the true correspondences
# (oracle): its angle about the image x axis, the cosine between the worm's x axis and the atlas's,
# and the determinant of the linear part. The atlas is in the frame of its initial worm.
rows = []
for j, w in enumerate(worms):
    labs, F = feats[j]
    r = runs[("oracle", True)][j]
    names_j = r["names"]
    train = [f for i, f in enumerate(feats) if i != j]
    n_, mu_j, sig_j, inf_j = npl.fit_atlas(train)
    P = F[:, :3]
    c, m = npl._pca(P)
    Q = (P - m) @ c
    rows.append(dict(subject=w["subject"], view=w["subject"][-2:], atlas_initial_worm=[f for i, f in enumerate(worms) if i != j][inf_j["init_worm"]]["subject"],
                     pc_extent_um=np.round(Q.max(0) - Q.min(0), 1).tolist(), image_extent_um=np.round(P.max(0) - P.min(0), 1).tolist(),
                     **{k: round(v, 2) for k, v in npl.oracle_rotation(F, mu_j, sig_j, r["true"]).items()},
                     acc_methods=100 * npl.scores([runs[("methods", True)][j]])[0][0], acc_code=100 * npl.scores([runs[("code", True)][j]])[0][0],
                     acc_rotated=100 * npl.scores([runs[("rotated", True)][j]])[0][0], rotated_start_deg=runs[("rotated", True)][j]["info"]["start_deg"],
                     acc_oracle=100 * npl.scores([runs[("oracle", True)][j]])[0][0]))
orient = pd.DataFrame(rows)
print(orient.round(1).to_string(index=False))
print("\nMean per-neuron accuracy (%) by view:")
print(orient.groupby("view")[["acc_methods", "acc_code", "acc_rotated", "acc_oracle"]].mean().round(1).to_string())

# The rotation search of the CELL_ID alignment: distinct starting configurations over theta and sign
labs, F = feats[0]
n_, mu0, sig0, _ = npl.fit_atlas(feats[1:])
c_at, m_at = npl._pca(mu0[:, :3]); c_at = c_at * np.linalg.det(c_at)
coeff_mu, mu_mu = npl._pca((mu0[:, :3] - m_at) @ c_at); coeff_mu = coeff_mu * np.linalg.det(coeff_mu)
starts = []
for sgn in (+1, -1):
    for theta in npl.THETAS:
        pos = npl.major_axis_align(F[:, :3], sgn)
        R = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
        pos[:, 1:3] = pos[:, 1:3] @ R
        starts.append(np.round(npl.major_axis_align(pos @ coeff_mu.T + mu_mu, 1), 6))
distinct = len({s.tobytes() for s in starts})
print(f"\nCELL_ID rotation search, worm {worms[0]['subject']}: {len(starts)} candidates (2 signs x {len(npl.THETAS)} angles) "
      f"give {distinct} distinct starting configurations")

### fig7bc-identification-accuracy

> Unsupervised identification of neurons against the statistical atlas of NeuroPAL colors and
> positions is 86% accurate in the head and 94% in the tail, cross-validated.

Expected: head 86 +/- 8.6%, tail 94 +/- 9.4%. Readings of the number: (i) mean of per-worm
accuracy, (ii) pooled, each (1) per neuron or (2) by class for left/right pairs.

In [ ]:
tab = pd.DataFrame([dict(alignment=a, **score_rows(runs[(a, True)])) for a in ("methods", "code", "rotated", "oracle")])
print("Head accuracy (%), position and color:")
print(tab.round(2).to_string(index=False))
print("\nWithin 86 +/- 8.6 (77.4 to 94.6), on the unrounded values:")
print((tab.set_index("alignment").apply(lambda c: c.between(77.4, 94.6))).to_string())
per = pd.DataFrame({a: 100 * npl.scores(runs[(a, True)])[0] for a in ("methods", "code", "rotated", "oracle")},
                   index=[w["subject"] for w in worms]).round(1)
print("\nPer held-out worm (per-neuron accuracy, %):")
print(per.to_string())
print("\nAlgorithm 5 converged in every worm:", all(r["info"]["converged"] for r in runs[("methods", True)]),
      "; iterations:", [r["info"]["iterations"] for r in runs[("methods", True)]])
print("Rotated variant: start kept per worm (degrees):", [r["info"]["start_deg"] for r in runs[("rotated", True)]],
      "; converged in every worm:", all(r["info"]["converged"] for r in runs[("rotated", True)]))
print("Chance (one of about 197 identities): 0.5%")

rows = []
for a in ("methods", "code", "rotated"):
    r = npl.run_loo_detections(feats, worms, detections, a)
    acc = np.array([x["hits"] / x["named"] for x in r]); cacc = np.array([x["class_hits"] / x["named"] for x in r])
    rows.append(dict(alignment=a, mean_per_worm_neuron=100 * acc.mean(),
                     pooled_neuron=100 * sum(x["hits"] for x in r) / sum(x["named"] for x in r),
                     mean_per_worm_class=100 * cacc.mean(),
                     pooled_class=100 * sum(x["class_hits"] for x in r) / sum(x["named"] for x in r),
                     annotations_matched_pct=100 * sum(x["matched"] for x in r) / sum(x["named"] for x in r)))
print("\nReading b, automatic detections (head, %):")
print(pd.DataFrame(rows).round(1).to_string(index=False))

In [ ]:
rows = []
view = np.array([w["subject"][-2:] for w in worms])
for label, kw in [("main", {}), ("5 alternations", dict(atlas_iter=5)), ("100 alternations", dict(atlas_iter=100)),
                  ("seed 1 (initial worm)", dict(seed=1)), ("no gauge, 20 alternations", dict(gauge=False))]:
    for a in ("methods", "code", "rotated", "oracle"):
        res_ = runs[(a, True)] if label == "main" else npl.run_loo(feats, a, **kw)
        per_ = 100 * npl.scores(res_)[0]
        rows.append(dict(atlas=label, alignment=a, initial_worms=sorted({[f for i, f in enumerate(worms) if i != j][r["atlas_info"]["init_worm"]]["subject"] for j, r in enumerate(res_)}),
                         **score_rows(res_), mean_LR_worms=per_[view == "LR"].mean(), mean_DV_worms=per_[view == "DV"].mean()))
print("Variants of the atlas fit (head, position and color, %); the last two columns are per-neuron means over the five worms of each view:")
print(pd.DataFrame(rows).round(2).to_string(index=False))

**Result: not reproduced for the head; the tails are not in the dandiset.** No reading of the
paper's two alignments reaches 77.4%: on the annotated neurons Algorithm 5 as written gives 39.0 to
51.8% over the four readings of the number and the CELL_ID alignment 55.0 to 63.2%; on the
automatic detections, 21.4 to 27.6% and 24.8 to 30.7%. The atlas variants give 38.7 to 67.0% with
these two alignments. Much of the shortfall is orientation: Algorithm 5 gets 9.6% on average on
the five dorsal-ventral heads, which are stored rotated by 92.6 to 102.7 degrees about the long axis
relative to the atlas, against 68.5% on the left-right heads, and the CELL_ID search produces 4
distinct starts from its 52 candidates on the worm checked. Started from four rotations (not the
paper's method), it gives 74.7 to 77.4%: the highest reading, 77.38% by class with the mean over
worms, misses the bound of 77.4% by 0.02 points. Under the four variants of the atlas fit the
rotated starts give 73.0 to 79.4% by class and 68.1 to 70.8% per neuron. Both class readings are
inside the tolerance with 5 alternations (79.40 and 79.35%) and with the other initial worm (78.76
and 78.72%), only the mean-per-worm class reading with no gauge (77.43%, pooled 77.35%), and
neither with 100 alternations (73.13 and 73.04%); no per-neuron reading is inside under any
variant. So with orientation undone, which the paper's method does not do here, the outcome would
turn on the atlas fit and the reading by a few hundredths of a point. Aligning each held-out worm by its true correspondences, not the
paper's method, gives 77.7 to 80.2%, so even a perfect alignment of this atlas lies at the lower
edge of the tolerance. Algorithm 5 converged in every worm under both starts.

### fig7bc-ventral-ganglion-lowest

> Accuracy depends on neuron density; it is high for all tail ganglia and lower for the dense
> ventral ganglion of the head.

Blocked: the criterion compares every tail ganglion with the head's ventral ganglion, and the
dandiset holds no tails. The head part the criterion asks to report (the rank of the ventral
ganglion among head ganglia, and the ganglion densities) is shown, with ganglion membership from
the CELL_ID neuron tables (Data/NeuroPAL/herm_data.mat at d9f5df9), as the criterion directs.


In [ ]:
from scipy.spatial import ConvexHull, cKDTree

gang = npl.load_ganglia()
print("Atlas neurons with no ganglion in the CELL_ID table:", [n for n in names if n not in gang])
hpos = mu[:, :3]
tree = cKDTree(hpos)
nn = tree.query(hpos, k=2)[0][:, 1]
dens = []
for g in npl.HEAD_GANGLIA:
    idx = [i for i, n in enumerate(names) if gang.get(n) == g]
    vol = ConvexHull(hpos[idx]).volume if len(idx) >= 4 else np.nan
    dens.append(dict(ganglion=g, atlas_neurons=len(idx), hull_volume_um3=vol, neurons_per_1000_um3=1000 * len(idx) / vol,
                     median_nearest_neighbour_um=np.median(nn[idx])))
dens = pd.DataFrame(dens).set_index("ganglion")
for a in ("methods", "code", "rotated", "oracle"):
    hits = {}
    for r in runs[(a, True)]:
        for lab, k in zip(r["labels"], r["assign"]):
            hits.setdefault(gang.get(lab, "none"), []).append(r["names"][k] == lab)
    dens[f"acc_{a}"] = [100 * np.mean(hits[g]) for g in dens.index]
    dens[f"rank_{a}"] = dens[f"acc_{a}"].rank(ascending=True).astype(int)
print("Head ganglia: density in the atlas, and pooled per-neuron accuracy over the ten held-out worms (%); rank 1 = lowest accuracy")
print(dens.round(2).to_string())
print("\nSpearman correlation over the seven head ganglia between accuracy and density (neurons per 1000 um^3):",
      {a: round(float(stats.spearmanr(dens[f"acc_{a}"], dens.neurons_per_1000_um3)[0]), 2) for a in ("methods", "code", "rotated", "oracle")})

**Result: blocked, the tails are not in the dandiset.** In the head, the ventral ganglion has the
lowest accuracy of the seven head ganglia under every alignment (12.9% with Algorithm 5, 22.1% with
the CELL_ID alignment, 29.3% with the rotated starts and 31.6% with the true correspondences,
while the other six range from 39.7 to 63.1% with Algorithm 5 and from 62.9 to 96.9% with the
rotated starts). By the median distance to the nearest atlas neuron
(1.87 um) it is the second densest after the dorsal ganglion (1.56 um, 6 neurons), and by
neurons per convex-hull volume the third. Accuracy does not follow density across the seven head
ganglia (Spearman rho -0.25 to 0.04).

### ms1-position-only-accuracy

> Restricted to neuron positions without color, automated identification accuracy drops to 50%
> in the head and 68% in the tail.

Expected: head 50 +/- 5%, tail 68 +/- 6.8%; each below the full model under the same reading.

In [ ]:
tabp = pd.DataFrame([dict(alignment=a, **score_rows(runs[(a, False)])) for a in ("methods", "code", "rotated", "oracle")])
print("Head accuracy (%), position only:")
print(tabp.round(2).to_string(index=False))
print("\nWithin 50 +/- 5 (45 to 55):")
print(tabp.set_index("alignment").apply(lambda c: c.between(45, 55)).to_string())

# The rotated starts (not the paper's method) under the variants of the atlas fit, position only
rows = []
for label, kw in [("main", {}), ("5 alternations", dict(atlas_iter=5)), ("100 alternations", dict(atlas_iter=100)),
                  ("seed 1 (initial worm)", dict(seed=1)), ("no gauge, 20 alternations", dict(gauge=False))]:
    res_ = runs[("rotated", False)] if label == "main" else npl.run_loo(feats, "rotated", color=False, **kw)
    rows.append(dict(atlas=label, **score_rows(res_)))
print("\nPosition only, rotated starts, variants of the atlas fit (head, %):")
print(pd.DataFrame(rows).round(2).to_string(index=False))
print("\nBelow the full model under the same reading and alignment:")
print((tabp.set_index("alignment") < tab.set_index("alignment")).to_string())

**Result: not reproduced for the head; the tails are not in the dandiset.** Position-only
accuracy is 20.4 to 22.8% with Algorithm 5 and 24.3 to 27.1% with the CELL_ID alignment, below
45%, though below the full model under every reading as the claim says. With the rotated starts
(not the paper's method) it is 43.1 to 44.0%, still below 45%; under the variants of the atlas fit
the rotated starts give 45.1 to 46.7% with 5 alternations, the other initial worm and no gauge,
inside the tolerance, and 37.8 to 38.8% with 100 alternations. With alignment by the true
correspondences (not the paper's method either) it is 48.7 to 49.4%, inside 45 to 55%.

### fig7b-eight-labels-above-90

> Manually annotating eight neurons, chosen by the software as the least certain, raises head
> identification accuracy above 90% on average.

The least certain neuron is the one whose Sinkhorn row is closest to uniform, read as (a) highest
entropy or (b) lowest top probability. Accuracy after eight is over all neurons (annotated ones
correct) or over the unannotated ones.

In [ ]:
guided = {}
t0 = time.time()
for a in ("methods", "code", "rotated"):
    for order in ("entropy", "top"):
        guided[(a, order)] = [npl.guided_annotation(r, 50, order) for r in runs[(a, True)]]
print(f"guided annotation in {time.time() - t0:.0f} s")
rows = []
for (a, order), g in guided.items():
    all8 = np.array([x[0][7] for x in g]); rest8 = np.array([x[1][7] for x in g])
    rows.append(dict(alignment=a, ranking=order, start=100 * npl.scores(runs[(a, True)])[0].mean(),
                     after8_all=100 * all8.mean(), after8_unannotated=100 * rest8.mean(),
                     worms_above_90_all=int((all8 > 0.9).sum()), best_worm_after8_all=100 * all8.max()))
print(pd.DataFrame(rows).round(1).to_string(index=False))
print("\nPer worm, code alignment, entropy ranking, accuracy over all neurons after 8 (%):",
      np.round([100 * x[0][7] for x in guided[('code', 'entropy')]], 1).tolist())

**Result: not reproduced.** After eight annotations of the least certain neurons the mean head
accuracy is 43.0 to 43.1% (all neurons) with Algorithm 5 and 58.7 to 58.8% with the CELL_ID
alignment, and no held-out worm exceeds 90% (the best worm reaches 83.0% with Algorithm 5 and 86.2
to 86.7% with the CELL_ID alignment); it starts from 39.1 and 55.0%. With the rotated starts (not
the paper's method) it is 78.4 to 79.3%, from 74.8%, and one worm exceeds 90% (91.5%) under the
entropy ranking.

### ms1-uncertain-first-faster

> Annotating the most uncertain neurons first raises identification accuracy much faster than
> annotating neurons in random order, in head and tail.

1 to 50 annotations in the head, against 100 random orders per held-out worm.

In [ ]:
rng = np.random.default_rng(715)
rand = {}
t0 = time.time()
for a in ("methods", "code", "rotated"):
    rand[a] = np.array([np.mean([npl.guided_annotation(r, 50, "random", rng)[0] for _ in range(100)], axis=0)
                        for r in runs[(a, True)]])
print(f"random orders in {time.time() - t0:.0f} s")
rows = []
for a in ("methods", "code", "rotated"):
    rnd = rand[a].mean(0)
    for order in ("entropy", "top"):
        unc = np.mean([x[0] for x in guided[(a, order)]], axis=0)
        rows.append(dict(alignment=a, ranking=order, counts_above_random=int((unc > rnd).sum()),
                         of=len(rnd), mean_uncertain=100 * unc.mean(), mean_random=100 * rnd.mean(),
                         at_8_uncertain=100 * unc[7], at_8_random=100 * rnd[7],
                         at_50_uncertain=100 * unc[49], at_50_random=100 * rnd[49]))
print(pd.DataFrame(rows).round(1).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5), sharey=True)
k = np.arange(1, 51)
lo, hi = 1.0, 0.0
for ax, a in zip(axes, ("methods", "code", "rotated")):
    for order, c in (("entropy", "C0"), ("top", "C1")):
        y = np.mean([x[0] for x in guided[(a, order)]], axis=0)
        ax.plot(k, 100 * y, color=c, label=f"most uncertain first ({order})")
        lo, hi = min(lo, y.min()), max(hi, y.max())
    ax.plot(k, 100 * rand[a].mean(0), color="0.4", label="random order")
    lo, hi = min(lo, rand[a].mean(0).min()), max(hi, rand[a].mean(0).max())
    ax.set_title(f"{a} alignment"); ax.set_xlabel("annotated neurons")
axes[0].set_ylabel("head accuracy, all neurons (%)"); axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()
print(f"Plotted range {100 * lo:.1f} to {100 * hi:.1f}%; no point outside the axes")

**Result: partially reproduced: the head half holds, and the tail half cannot be tested, as the
tails are not in the dandiset.** With the most uncertain neurons first, the mean accuracy is above
the random-order curve at all 50 annotation counts, under both rankings and all three alignments,
and its mean over the range is higher (68.0 against 62.4% with the CELL_ID alignment, 52.2 against
48.5% with Algorithm 5, 86.8 against 79.0% with the rotated starts; at 50 annotations 79.1 against
69.5% with the CELL_ID alignment). The gain at eight annotations is small: 58.8 against 57.3% with
the CELL_ID alignment, 43.0 against 42.1% with Algorithm 5 and 79.3 against 76.0% with the rotated
starts.

### ms1-calibration

> The identification's confidence is calibrated; per-neuron confidence and accuracy largely agree
> in head and tail.

Per atlas neuron, confidence is the mean over held-out worms of the Sinkhorn probability of the
assigned identity and accuracy the fraction of worms in which it is right. Readings: (a) positive
Spearman correlation with p < 0.05, (b) mean absolute difference under 10 points. Left/right pairs
are scored per neuron and, as a variant, by class (the upper bound of footnote 1 of MS1 6.4).

In [ ]:
def calibration(res, by_class):
    rows = {}
    for r in res:
        names = r["names"]
        nset = set(names) | set(r["labels"])
        cls = np.array([npl.pair_class(n, nset) for n in names])
        for k, (lab, a) in enumerate(zip(r["labels"], r["assign"])):
            if by_class:
                conf = r["P"][k, cls == cls[a]].sum()
                hit = cls[a] == npl.pair_class(lab, nset)
            else:
                conf = r["P"][k, a]
                hit = names[a] == lab
            rows.setdefault(lab, []).append((conf, hit))
    df = pd.DataFrame({n: np.mean(v, axis=0) for n, v in rows.items()}, index=["confidence", "accuracy"]).T
    rho, p = stats.spearmanr(df.confidence, df.accuracy)
    mad = 100 * np.abs(df.confidence - df.accuracy).mean()
    return df, dict(neurons=len(df), mean_conf=100 * df.confidence.mean(), mean_acc=100 * df.accuracy.mean(),
                    spearman_rho=rho, spearman_p=p, mean_abs_diff_points=mad,
                    a_holds=bool(rho > 0 and p < 0.05), b_holds=bool(mad < 10))

rows = []
cal = {}
for a in ("methods", "code", "rotated"):
    for by_class in (False, True):
        df, s = calibration(runs[(a, True)], by_class)
        cal[(a, by_class)] = df
        rows.append(dict(alignment=a, by_class=by_class, **s))
cal_tab = pd.DataFrame(rows)
cal_tab["spearman_p"] = cal_tab["spearman_p"].map(lambda p: f"{p:.1e}")
print(cal_tab.round(2).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, a in zip(axes, ("methods", "code")):
    df = cal[(a, False)]
    ax.scatter(100 * df.confidence, 100 * df.accuracy, s=8, color="0.3")
    ax.plot([0, 100], [0, 100], color="C3", lw=1)
    ax.set_xlim(-2, 102); ax.set_ylim(-2, 102)
    ax.set_xlabel("confidence (%)"); ax.set_ylabel("accuracy (%)"); ax.set_title(f"{a} alignment, per neuron")
plt.tight_layout(); plt.show()
print("All points lie within 0 to 100% on both axes")

**Result: blocked, the two readings disagree.** Per-neuron confidence and accuracy correlate
(Spearman rho 0.51 to 0.70, p below 1e-13 in every run), so reading a holds, but confidence
(87 to 89% on average) exceeds accuracy (38 to 62%) by 28 to 49 points on average, so reading b
fails. With the rotated starts (not the paper's method) rho is 0.81 to 0.82 and the difference
16.3 to 17.7 points, so reading b still fails. The criterion blocks the claim when the readings disagree. The overconfidence follows from
the low accuracy of fig7bc-identification-accuracy here.

## Color violations

### ms2-neuropal-color-violations

> Using the atlas of neuron positions from the 10 worms to define neighbors, NeuroPAL's designed
> color map has a theoretical color margin violation of 3.5% over neighboring neuron pairs.

Neighbors from the head atlas fit on all ten worms (computeProbAdjacency); the NeuroPAL colors of
Optimal_Coloring `data.mat`, scaled by each channel's maximum and then to sum to one per neuron, as
find_optimal_colors.m does; margin 1/3. The code's measure, the mean shortfall below the margin
over neighboring pairs (find_optimal_colors.m), decides the claim; the Methods measure, the share
of neighboring pairs closer than the margin (Methods S2 section 4.3), is a variant. The claims file
was amended to this order after a reader's answer (questions/000715.yaml); it had the Methods
measure as the main result. As a reference, the same two measures are computed on the paper's own
inputs in `data.mat` (its 235-neuron head and tail atlas and its adjacency matrix).

In [ ]:
OC_URL = "https://raw.githubusercontent.com/Eviatar/Optimal_Coloring/990c597d1a7b0c38c4e7e99f8e8b60da411b3493/data.mat"
oc = sio.loadmat(io.BytesIO(requests.get(OC_URL, timeout=120).content), squeeze_me=True, struct_as_record=False)
oc_names = [str(x) for x in oc["neurons"]]
C = oc["NeuroPAL_colors"].astype(float)
C = np.nan_to_num(C / C.max(0))
Cn = C / C.sum(1, keepdims=True)
print(f"data.mat: {len(oc_names)} neurons with NeuroPAL colors; rows with no color: {(C.sum(1) == 0).sum()}")
COLS = ["code_shortfall_pct", "methods_share_pct", "adjacent_ordered_pairs"]
viol_cols = lambda v: (v[1], v[0], v[2])        # the code's measure first: it decides the claim
shared = [n for n in names if n in oc_names]
print(f"Head atlas neurons with a NeuroPAL color: {len(shared)} of {len(names)}; without: {[n for n in names if n not in oc_names]}")
ia = [names.index(n) for n in shared]; ic = [oc_names.index(n) for n in shared]
A_head = npl.prob_adjacency(mu[ia, :3], sigma[ia][:, :3, :3])
rows = [dict(neighbors="dataset head atlas (main)", **dict(zip(COLS,
                                                         viol_cols(npl.color_violations(Cn[ic], A_head)))))]
A_ref = oc["neuron_prob_distances"].astype(int)
rows.append(dict(neighbors="data.mat adjacency, 235 head and tail (paper's input)", **dict(zip(COLS, viol_cols(npl.color_violations(Cn, A_ref))))))
rows.append(dict(neighbors="data.mat adjacency, same head neurons", **dict(zip(COLS, viol_cols(npl.color_violations(Cn[ic], A_ref[np.ix_(ic, ic)]))))))
A_ref2 = npl.prob_adjacency(oc["neuron_positions"], np.transpose(oc["neuron_covariances"], (2, 0, 1)))
rows.append(dict(neighbors="this code's adjacency on the data.mat atlas", **dict(zip(COLS, viol_cols(npl.color_violations(Cn, A_ref2))))))
for label, kw in [("head atlas, initial worm 38_YAaDV", dict(init=2)), ("head atlas, 100 alternations", dict(max_iter=100)), ("head atlas, no gauge", dict(gauge=False))]:
    n_, m_, s_, i_ = npl.fit_atlas(feats, **kw)
    ja = [n_.index(n) for n in shared]
    rows.append(dict(neighbors=label, **dict(zip(COLS, viol_cols(npl.color_violations(Cn[ic], npl.prob_adjacency(m_[ja, :3], s_[ja][:, :3, :3])))))))
viol = pd.DataFrame(rows)
viol[["code_shortfall_pct", "methods_share_pct"]] *= 100
print(viol.round(2).to_string(index=False))
print(f"\nThis code's adjacency and data.mat's agree on {100 * (A_ref2 == A_ref).mean():.1f}% of the 235 x 235 entries.")

**Result: reproduced.** With neighbors from the head atlas of the dandiset, the code's measure,
the mean shortfall below the margin over neighboring pairs, is 3.64%, inside 3.5 +/- 0.35%, and
3.62 to 3.66% under the atlas variants. On the paper's own inputs in data.mat it is 3.47%. The
Methods share of neighboring pairs closer in color than the margin, a variant, is 21.50% on the
dataset's head atlas and on data.mat's inputs alike, far from 3.5%. The reader's answer moved the
code's measure to the main result; under the earlier criterion the claim was not reproduced.

## Summary

| Claim | Outcome |
|---|---|
| fig2cd-variability-differs | partially reproduced (head half holds; tails not in the dandiset) |
| fig2c-ollr-more-variable-than-olll | reproduced |
| ms1-detection-accuracy | partially reproduced (head only) |
| fig7bc-identification-accuracy | not reproduced (head only) |
| fig7bc-ventral-ganglion-lowest | blocked: tails not in the dandiset (in the head the ventral ganglion is lowest) |
| ms1-position-only-accuracy | not reproduced (head only) |
| fig7b-eight-labels-above-90 | not reproduced |
| ms1-uncertain-first-faster | partially reproduced (head half holds; tails not in the dandiset) |
| ms1-calibration | blocked: the criterion's readings disagree |
| ms2-neuropal-color-violations | reproduced (head atlas, with the paper's NeuroPAL colors) |
| fig7de-optimal-coloring-violations | tested in `optimal_coloring.ipynb` |
| claims on figures 1, 3 to 6, S1 and S2 | not tested in this notebook; see the record |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

How the analysis differs from the paper's: only heads are in the dandiset; the atlas fit of
Algorithm 4 does not converge on these data, and the atlas here is kept in the initial worm's frame
and stopped after 20 alternations (an assumption, with variants shown); the colors are read from
the volumes with the readout of Methods S1, since the files hold no per-neuron colors; the
detection is a Python port of the paper's MATLAB code with a different optimiser. The five heads
imaged in dorsal-ventral view are stored rotated by about 90 degrees about the long axis relative
to the five in left-right view, which Algorithm 5's identity start cannot undo; a variant started
from four rotations (not the paper's method) is shown for every identification claim. It changes
no outcome on the main atlas, where fig7bc-identification-accuracy misses its bound by 0.02 points
and ms1-position-only-accuracy by 1 point, but under three of the four variants of the atlas fit
it puts a class reading of fig7bc-identification-accuracy, and the position-only accuracy, inside
the tolerance.

What was found about the dataset: the files are the heads of the paper's ten otIs669 atlas worms,
identical to the Zenodo volumes, but their metadata name the strain OH16230, a spinning disk
confocal, a microfluidic chip and dates in 2021; the channel labelled GFP-GCaMP holds the
transmitted-light channel; the tails are missing; neuron names are stored as one character per
element of a ragged array; about 3% of the stored neurons have no name; and the files carry no
per-neuron colors and no orientation of the worm, though half the heads are stored in another
orientation from the other half.